> **SUPERSEDED (September 2026) - kept as the record of the ORIGINAL model (v1).** A later audit found three problems with this notebook's setup:
> (1) the rainfall input carried up to 30 minutes of rain that had **not yet fallen** (rainfall file `j` is the rain falling *during* `[1800 j, 1800 (j+1))` s, but each input frame
> received the file of the interval *containing* it); (2) a trivial per-pixel linear model beats this CNN-LSTM, and a tiny per-pixel/local network scores far higher;
> (3) this model's dense bottleneck could not use the depth history, so its "rainfall drives the forecast" finding is an artefact of that weakness.
> The current, corrected study is **`flood_forecasting_study_colab.ipynb`** (see `PROJECT_LOG.md`, "Completion, round 2"). Numbers printed here describe v1 only.

# CNN-LSTM Flood Forecasting - Real-Scale Training (Colab)

Trains the pipeline at full size on a Colab GPU (T4 is enough: ~10 s per epoch), then evaluates it, visualizes it
and runs a cross-regional test. Every cell of this notebook was executed locally at tiny size first
(`dryrun_notebook.py`); the first full-size run happened on Colab and its numbers are recorded in
`PROJECT_LOG.md`. **Runtime -> Change runtime type -> GPU** first.

## Read this before trusting any number

A local dry-run of the first draft of this notebook found problems that would have produced
misleading results. Each is fixed; the table says what and why.

| Finding (measured on the real data) | What changed |
|---|---|
| **Copying the last frame is almost exact**: persistence RMSE is 2 mm at the native 5-min step (13 mm at a 30-min stride). Thresholding the last frame's max depth predicts severity **99.9%** correctly (96.7% at 30 min). | Every depth and severity result is reported **next to these baselines**. A model that doesn't beat them has learned nothing useful, however small its error looks. |
| The decoder rebuilt the whole depth field from a 128-number vector with no link to the observed frame - it could never reach a near-exact copy. (This, not only "too little data", is why the Stage 2 predictions were blurry.) | **Persistence-anchored decoder**: output = last observed frame + a learned change, zero-initialised so an untrained model *is* the persistence forecast. |
| At the 5-min step, 5 history frames span 25 min, but rainfall is only sampled every 30 min - the rainfall channel was effectively constant. | **`TIME_STRIDE = 6`** (30 min): each step sees its own rainfall value; horizon is 0.5-2.5 h ahead. |
| Severity (max depth in a 128px patch) is **not scale-invariant**: UK is 100% severe, Pakistan ~100%, Mozambique 84%; only Australia has all four classes. | Severity is trained/validated on **Australia only**, and is **not scored on the UK** (an "always severe" model would look perfect). |
| The 60m / 480m depth rasters are resampled, cropped copies of the 30m grid while DEM/rainfall/land-use stay on the 30m grid; UK's layers are offset by 5px. | Every region/resolution has a **measured, cross-checked geometry** (`LAYER_GEOMETRY`); one without it is refused. Pakistan, Mozambique and the 60m data are covered by `flood_forecasting_multires_colab.ipynb`. |
| First-draft sampler never reached the second region; split was random over windows from the *same* patches (leaks). | Random sampling per region; **split by spatial position** - validation patches are never seen in training. |
| Raw inputs: DEM in hundreds of metres next to rainfall in mm. | Fixed normalization inside the model; DEM is patch-relative so it can transfer across regions. |

**This notebook: train on Australia, test on the UK at 30m.** The paper's other two transfer tests -
Australia -> UK at 60m and Pakistan -> Mozambique at 480m - are in `flood_forecasting_multires_colab.ipynb`.
Mixing resolutions in one training set is deliberately not attempted: a 128px patch spans 3.8 km, 7.7 km or
61 km depending on the resolution. The severity task is also close to trivial under this label definition; see below.

## Setup

In [ ]:
!pip install -q huggingface_hub rasterio scipy folium
import warnings
import tensorflow as tf
import rasterio
warnings.filterwarnings("ignore", category=rasterio.errors.NotGeoreferencedWarning)
print("GPU available:", tf.config.list_physical_devices('GPU'))
print("TensorFlow version:", tf.__version__)

## Configuration

Edit here. Defaults suit a Colab GPU session; reduce `PER_CLASS_TRAIN` / `EPOCHS` for a quick first pass.

In [ ]:
PATCH_SIZE = 128
T_IN, HORIZON = 5, 5            # 5 observed steps -> 5 forecast steps
TIME_STRIDE = 6                 # depth steps between frames: 6 x 5 min = 30 min (rainfall's native resolution)
PER_CLASS_TRAIN = 100           # class-balanced training windows per severity class
PER_CLASS_VAL = 60              # class-balanced validation windows per class (25 gave +-4.6 points of noise on accuracy)
EPOCHS = 40
LEARNING_RATE = 1e-3            # peak; decays on a cosine schedule to 2% of this by the last step
CLIPNORM = 1.0                  # gradient clipping; batch size 1 makes single steps noisy
N_START_INDICES = 80           # time starts scanned when building Australia's window pool
N_UNIFORM_AU = 150              # unbalanced Australia val windows, for the like-for-like comparison with UK
UK_START_STEP = 12              # UK: every 12th start index (x 2 spatial patches)
CENSUS_START_STEP = 24          # start-index step for the severity-class census
VAL_POSITION_FRACTION = 0.25    # fraction of Australia's patches held out entirely for validation
DEM_NODATA_STD = 1.0            # skip patches whose DEM is (near-)flat: outside the simulation domain
SEED = 42
AUGMENT = False                 # random flips / 90-degree rotations of training windows. OFF: helped severity on 2 of 2 seeds but destabilised depth on one (PROJECT_LOG.md)

## Pull the dataset from Hugging Face

Only Australia and UK, and only their own DEM / land-use / rainfall files. **This is still large:
about 13.5GB in 4,385 files, almost all of it Australia's 2,881 depth frames (4.6MB each)** - so expect this
first cell to be the slowest part of the notebook (measured on Colab: 7 min 43 s).
Colab's disk is ephemeral, so this re-downloads each fresh session unless you cache it to Drive.

In [ ]:
from huggingface_hub import snapshot_download

DATA_ROOT = snapshot_download(
    repo_id="mahajananhad/floodcastbench",
    repo_type="dataset",
    allow_patterns=[
        "High-fidelity flood forecasting/30m/Australia/*",
        "High-fidelity flood forecasting/30m/UK/*",
        "Relevant data/DEM/Australia_DEM.*",
        "Relevant data/DEM/UK_DEM.*",
        "Relevant data/Land use and land cover/Australia.*",
        "Relevant data/Land use and land cover/UK.*",
        "Relevant data/Rainfall/Australia flood/*",
        "Relevant data/Rainfall/UK flood/*",
        "Relevant data/Georeferenced files/*",
    ],
    max_workers=16,  # thousands of small files: more parallel downloads help
)
print("Downloaded to:", DATA_ROOT)

## Core modules (verified locally)

In [ ]:
%%writefile real_data_loader.py
"""
Real FloodCastBench data loader for one region (Stage 1 of the build plan).

Loads a spatial patch + temporal window of depth, DEM, and rainfall for a
single region, aligns rainfall's separate 30-minute calendar-date series to
depth's 5-minute simulation-second series, and derives a severity label from
future depth using the fixed thresholds decided in the project plan.

Verified against the real downloaded files:
  - depth: <region>/<t>.tif, t = unpadded seconds since sim start, step 300s
  - rainfall: Relevant data/Rainfall/<region> flood/<YYYYMMDD>-S<HHMMSS>.tif, step 1800s
  - Australia: depth, rainfall, DEM and land use all share the same 1073x1073 grid.
  - UK: depth is 170x275 but the other layers are 180x285 - a measured (5, 5) pixel
    offset is applied (see LAYER_GEOMETRY).
  - 60 m and 480 m depth rasters (Australia/UK at 60 m; Pakistan/Mozambique at 480 m) are the 30 m
    simulation resampled to 2x2 / 16x16 base-pixel cells and cropped, while DEM / land use / rainfall
    stay on the 30 m base grid. LAYER_GEOMETRY records each combination's scale and crop offset
    (recovered from the data and cross-checked, see its comment); DEM / rainfall are block-averaged
    to the depth cells. A combination with no registered geometry is refused, never guessed.
    (An earlier version of this loader claimed all layers matched - checked only on Australia - and
    later refused Pakistan/Mozambique as "not alignable" - a failure of the first search, not a
    property of the data.)

CONFIRMED PHYSICALLY (completion/a_rain_lag.py, round 2): rainfall file j is the rain that FALLS DURING [1800 j, 1800 (j+1)) s of the simulation; the values are mm/h
intensities (depth gain over an interval = 0.494-0.502 x the file value; correlation 1.00 at lag 0 in all four regions). The archive itself gives no clock time for
t = 0 (no TIFF tag or readme ties it to a calendar date); none is used. Original reasoning: rainfall's first file corresponds to depth's t=0. Evidence for this: for all four events the rainfall series starts on the
paper's stated event start date, and its duration equals the depth duration to the second
(e.g. Australia: 481 files at 1800 s = 10 days = 864000 s of depth at 300 s). Date-level
agreement was checked first; the 30-minute alignment is now measured (see above).
"""
import os
import re
import glob
from datetime import datetime

import numpy as np
import rasterio

DEPTH_STEP_SECONDS = 300
RAINFALL_STEP_SECONDS = 1800

# Pixel offset (rows, cols) of the depth grid inside the DEM / land-use / rainfall grid,
# for regions where those layers are larger than the depth raster. Measured, not assumed:
# UK depth is 170x275 but its DEM/land-use/rainfall are 180x285. Both an independent
# depth-vs-DEM correlation search (best at dy=5,dx=5, r=-0.27 vs a max of +0.03 over all
# 121 candidate offsets) and a wet-pixels-on-baseline-water enrichment search (~5x at
# dy=5..6,dx=5..6) peak there - consistent with a centered 5px border crop on each side.
# Australia's layers all match the depth grid exactly (verified) so it needs no entry.
#
# Geometry of EVERY (region, resolution) relative to the 30 m base grid that the DEM / land use / rainfall live on:
#   (scale, (row_offset, col_offset))  = base pixels per depth cell, and the base-pixel position of depth cell (0, 0).
# The 60 m and 480 m depth rasters are the 30 m results resampled to 2x2 and 16x16 base-pixel cells and CROPPED (the paper:
# "resampled ... using bilinear interpolation"). Each geometry below was recovered from the data with a probe that was first
# validated on the known cases (Australia 30 m -> (0,0), UK 30 m -> (5,5)), and cross-checked by independent signals:
#   Mozambique 480 m  (16, (150, 50))  terrain-structure r=-0.27 (5.2 robust-sd), DEM NoData all outside the crop
#   Pakistan   480 m  (16, (42, 178))  terrain-structure r=-0.09 (5.5 robust-sd) AND baseline-water r=+0.10 agree; NoData outside
#   UK          60 m  (2,  (5, 5))     terrain-structure r=-0.44 at (5..6, 5..6)
#   Australia   60 m  (2,  (0, 0))     terrain-structure r=-0.26 at (0..1, 0..1)
# Precision is about +-2 base pixels (60 m), i.e. < 6% of a 480 m cell. Scripts and logs: verification_outputs/audit/.
# (An earlier version of this loader refused Pakistan and Mozambique as "not alignable"; that was a failure of the first
# search - it did not remove the large-scale terrain trend - not a property of the data.)
LAYER_GEOMETRY = {
    ("Australia", 30): (1, (0, 0)), ("UK", 30): (1, (5, 5)),
    ("Australia", 60): (2, (0, 0)), ("UK", 60): (2, (5, 5)),
    ("Pakistan", 480): (16, (42, 178)), ("Mozambique", 480): (16, (150, 50)),
}
LAYER_OFFSETS = {"UK": (5, 5)}  # kept for older callers: 30 m offsets only

# The DEM / rainfall rasters carry the float32 NoData sentinel (-3.4028235e38): the UK layers have a padding ring of it
# outside the depth-aligned crop. A window that touched it would feed ~1e38 into the network (and the flat-DEM filter
# in windows.py would count such a patch as "valid", its DEM std being enormous). None of the windows this project
# uses touch it (verification_outputs/audit/investigate_uk_nodata.py), but the loader now refuses instead of trusting that.
NODATA_BELOW = -1e30


def has_nodata(arr: np.ndarray) -> bool:
    """True if `arr` holds the NoData sentinel or any non-finite value."""
    return bool((arr < NODATA_BELOW).any() or not np.isfinite(arr).all())

# Fixed absolute depth thresholds (meters), decided in the project plan.
SEVERITY_THRESHOLDS = [0.3, 1.0, 2.0]  # -> none / minor / moderate / severe
SEVERITY_CLASSES = ["none", "minor", "moderate", "severe"]


def depth_to_severity(max_depth_m: float) -> int:
    """Bin a max-depth value (meters) into one of the 4 fixed severity classes."""
    idx = 0
    for t in SEVERITY_THRESHOLDS:
        if max_depth_m >= t:
            idx += 1
    return idx  # 0=none, 1=minor, 2=moderate, 3=severe


class RegionData:
    """Indexes one region's depth/rainfall/DEM files without loading pixel data."""

    def __init__(self, root_dir: str, region: str, fidelity_dir: str):
        """
        root_dir: path to the extracted FloodCastBench folder
        region: e.g. "Australia"
        fidelity_dir: e.g. "High-fidelity flood forecasting/30m"
        """
        self.root_dir = root_dir
        self.region = region

        depth_dir = os.path.join(root_dir, fidelity_dir, region)
        depth_files = glob.glob(os.path.join(depth_dir, "*.tif"))
        if not depth_files:
            raise FileNotFoundError(f"No depth TIFFs found in {depth_dir}")

        def depth_seconds(path):
            name = os.path.splitext(os.path.basename(path))[0]
            m = re.match(r"^(\d+)$", name)
            if not m:
                raise ValueError(f"Expected purely numeric depth filename, got: {path}")
            return int(m.group(1))

        self.depth_files = sorted(depth_files, key=depth_seconds)
        self.depth_seconds = [depth_seconds(p) for p in self.depth_files]

        step = self.depth_seconds[1] - self.depth_seconds[0]
        for i in range(1, len(self.depth_seconds)):
            gap = self.depth_seconds[i] - self.depth_seconds[i - 1]
            if gap != step:
                raise ValueError(
                    f"Depth sequence is not evenly spaced at index {i}: "
                    f"gap={gap}s, expected {step}s"
                )
        self.depth_step_seconds = step

        rain_dir = os.path.join(root_dir, "Relevant data", "Rainfall", f"{region} flood")
        rain_files = glob.glob(os.path.join(rain_dir, "*.tif"))
        if not rain_files:
            raise FileNotFoundError(f"No rainfall TIFFs found in {rain_dir}")

        def rain_datetime(path):
            name = os.path.splitext(os.path.basename(path))[0]
            m = re.match(r"^(\d{8})-S(\d{6})$", name)
            if not m:
                raise ValueError(f"Unexpected rainfall filename format: {path}")
            return datetime.strptime(m.group(1) + m.group(2), "%Y%m%d%H%M%S")

        rain_with_dt = sorted(((rain_datetime(p), p) for p in rain_files), key=lambda x: x[0])
        self.rain_files = [p for _, p in rain_with_dt]
        self.rain_start = rain_with_dt[0][0]
        rain_step = (rain_with_dt[1][0] - rain_with_dt[0][0]).total_seconds()
        for i in range(1, len(rain_with_dt)):
            gap = (rain_with_dt[i][0] - rain_with_dt[i - 1][0]).total_seconds()
            if gap != rain_step:
                raise ValueError(f"Rainfall sequence is not evenly spaced at index {i}")
        self.rain_step_seconds = rain_step

        self.dem_path = os.path.join(root_dir, "Relevant data", "DEM", f"{region}_DEM.tif")
        if not os.path.isfile(self.dem_path):
            raise FileNotFoundError(self.dem_path)

        self.landuse_path = os.path.join(root_dir, "Relevant data", "Land use and land cover", f"{region}.tif")
        if not os.path.isfile(self.landuse_path):
            raise FileNotFoundError(self.landuse_path)

        with rasterio.open(self.depth_files[0]) as src:
            self.grid_height, self.grid_width = src.height, src.width

        m = re.search(r"(\d+)m$", fidelity_dir.rstrip("/"))
        if not m:
            raise ValueError(
                f"Cannot parse pixel size from fidelity_dir={fidelity_dir!r} "
                f"(expected it to end in e.g. '30m')"
            )
        self.pixel_size_m = float(m.group(1))

        key = (region, int(round(self.pixel_size_m)))
        if key not in LAYER_GEOMETRY:
            raise ValueError(
                f"{region} at {self.pixel_size_m:.0f} m has no registered layer geometry in LAYER_GEOMETRY - windows read "
                f"at the same (row, col) would cover different geography. Refusing to load rather than return misaligned data."
            )
        self.layer_scale, self.layer_offset = LAYER_GEOMETRY[key]
        dy, dx = self.layer_offset
        s = self.layer_scale
        for label, path in (("DEM", self.dem_path), ("land use", self.landuse_path),
                            ("rainfall", self.rain_files[0])):
            with rasterio.open(path) as src:
                layer_h, layer_w = src.height, src.width
            if dy + s * self.grid_height > layer_h or dx + s * self.grid_width > layer_w:
                raise ValueError(
                    f"{region}: {label} grid is {layer_h}x{layer_w}, too small for the registered geometry "
                    f"(scale {s}, offset {self.layer_offset}) of a {self.grid_height}x{self.grid_width} depth grid."
                )

    def _layer_window(self, row: int, col: int, patch_size: int):
        """Window (in 30 m base pixels) into the DEM / land-use / rainfall grids for a depth-grid (row, col) patch."""
        dy, dx = self.layer_offset
        s = self.layer_scale
        return ((dy + s * row, dy + s * (row + patch_size)), (dx + s * col, dx + s * (col + patch_size)))

    def _read_layer(self, path: str, row: int, col: int, patch_size: int, nearest: bool = False) -> np.ndarray:
        """Read a base-grid layer for a depth-grid patch, reduced to (patch, patch): block-mean over the cell's
        scale x scale base pixels (nearest = the cell-centre pixel, for categorical land use)."""
        s = self.layer_scale
        with rasterio.open(path) as src:
            a = src.read(1, window=self._layer_window(row, col, patch_size), boundless=True, fill_value=0)
        if s == 1:
            return a
        a = a.reshape(patch_size, s, patch_size, s)
        if nearest:
            return a[:, s // 2, :, s // 2]
        return a.astype(np.float64).mean(axis=(1, 3)).astype(np.float32)

    def load_dem_patch(self, row: int, col: int, patch_size: int) -> np.ndarray:
        return self._read_layer(self.dem_path, row, col, patch_size).astype(np.float32)

    def rainfall_index_for_depth_second(self, depth_second: int) -> int:
        """Map a depth timestep (seconds since sim start) to its rainfall file index,
        holding each 30-min rainfall value constant across the depth steps inside it.
        NOTE (round 2): file j covers [1800 j, 1800 (j+1)) s, so this returns rain that has NOT YET FALLEN for a frame inside the interval (the 'hybrid' regime: partly
        future rain for the last observed frame). For past-only input use file floor(t/1800) - 1 ('causal'; see completion/engine.py)."""
        idx = int(depth_second // self.rain_step_seconds)
        return min(idx, len(self.rain_files) - 1)

    def window_frame_indices(self, start_idx: int, t_in: int, horizon: int, stride: int = 1):
        """Depth-file indices for one window: (history_indices, future_indices).

        History is start_idx, start_idx+stride, ...; the forecast continues at the same
        stride. stride=1 is the native 5-minute step; stride=6 is 30 minutes, which is
        rainfall's native resolution (at stride 1, five history frames span only 25
        minutes, so the rainfall channel is essentially constant across the sequence).
        The severity scan in windows.py uses this same method so scan and loader can
        never disagree about which frames a window covers."""
        history = [start_idx + stride * k for k in range(t_in)]
        future = [start_idx + stride * (t_in + k) for k in range(horizon)]
        if future[-1] >= len(self.depth_files):
            raise ValueError(
                f"start_idx={start_idx} t_in={t_in} horizon={horizon} stride={stride} runs past "
                f"the end of the depth sequence ({len(self.depth_files)} files)"
            )
        return history, future

    def max_start_index(self, t_in: int, horizon: int, stride: int = 1) -> int:
        """Largest start_idx for which a whole window still fits."""
        return len(self.depth_files) - 1 - stride * (t_in + horizon - 1)

    def load_patch_sequence(self, row: int, col: int, patch_size: int,
                             start_idx: int, t_in: int, horizon: int, stride: int = 1):
        """Read one (row, col) patch across the window described by window_frame_indices():
        the history frames are the model input, the future frames are both the regression
        targets and the source of the severity label.

        Returns: input_sequence (t_in, H, W, 3) [depth, DEM, rainfall (mm/h; for the last frame this includes up to 30 min of not-yet-fallen rain)],
                 depth_targets (horizon, H, W),
                 severity_label (int, 0-3), max_future_depth (float)
        """
        window = ((row, row + patch_size), (col, col + patch_size))
        history_idx, future_idx = self.window_frame_indices(start_idx, t_in, horizon, stride)

        dem_patch = self.load_dem_patch(row, col, patch_size)

        depth_frames = []
        rainfall_frames = []
        for i in history_idx:
            with rasterio.open(self.depth_files[i]) as src:
                depth_frames.append(src.read(1, window=window, boundless=True, fill_value=0).astype(np.float32))
            rain_idx = self.rainfall_index_for_depth_second(self.depth_seconds[i])
            rainfall_frames.append(self._read_layer(self.rain_files[rain_idx], row, col, patch_size).astype(np.float32))

        depth_hist = np.stack(depth_frames)          # (t_in, H, W)
        rainfall_hist = np.stack(rainfall_frames)     # (t_in, H, W)
        dem_seq = np.broadcast_to(dem_patch, depth_hist.shape)  # (t_in, H, W)
        input_sequence = np.stack([depth_hist, dem_seq, rainfall_hist], axis=-1)  # (t_in, H, W, 3)

        depth_targets = []
        for i in future_idx:
            with rasterio.open(self.depth_files[i]) as src:
                depth_targets.append(src.read(1, window=window, boundless=True, fill_value=0).astype(np.float32))
        depth_targets = np.stack(depth_targets)  # (horizon, H, W)

        for what, arr in (("depth history", depth_hist), ("rainfall", rainfall_hist),
                          ("DEM", dem_patch), ("depth targets", depth_targets)):
            if has_nodata(arr):
                raise ValueError(
                    f"{self.region}: {what} window at depth-grid (row={row}, col={col}) contains NoData/NaN pixels "
                    f"(the layers have a NoData padding ring outside the aligned crop). Refusing to return it."
                )

        max_future_depth = float(depth_targets.max())
        severity_label = depth_to_severity(max_future_depth)

        return input_sequence, depth_targets, severity_label, max_future_depth

    def load_landuse_patch(self, row: int, col: int, patch_size: int) -> np.ndarray:
        """Real Esri/Impact Observatory Sentinel-2 land-cover codes (verified against
        this exact product's published legend): 1=Water, 2=Trees, 4=Flooded Vegetation,
        5=Crops, 7=Built Area, 8=Bare Ground, 9=Snow/Ice, 10=Clouds, 11=Rangeland.
        At 60 m / 480 m the class of each cell's centre pixel is returned (nearest neighbour)."""
        return self._read_layer(self.landuse_path, row, col, patch_size, nearest=True).astype(np.int32)


In [ ]:
%%writefile capsule_layers.py
"""
Capsule Network layers (Sabour, Frosst & Hinton 2017, arXiv:1710.09829) for the
severity-classification branch: PrimaryCaps -> routing-by-agreement -> per-class
output capsules, whose vector lengths give the 4-way severity probability.

Kept as plain tf.keras.layers.Layer subclasses (not a separate framework) so the
whole model - depth-forecasting branch and severity branch alike - trains as one
TensorFlow/Keras graph: gradients from the severity loss reach back through
routing, through PrimaryCaps, into the SAME shared CNN encoder the LSTM branch
uses. This is the piece that was previously impossible when the capsule layer
lived in a separate PyTorch model (see project plan, Stage 3).
"""
import tensorflow as tf
from tensorflow.keras import layers


def squash(vectors: tf.Tensor, axis: int = -1, epsilon: float = 1e-7) -> tf.Tensor:
    """Capsule nonlinearity: shrinks short vectors to (near) zero and long vectors
    to just under length 1, so a capsule's output length can be read as a
    probability. squash(s) = (|s|^2 / (1 + |s|^2)) * (s / |s|)."""
    squared_norm = tf.reduce_sum(tf.square(vectors), axis=axis, keepdims=True)
    scale = squared_norm / (1.0 + squared_norm) / tf.sqrt(squared_norm + epsilon)
    return scale * vectors


class PrimaryCaps(layers.Layer):
    """First capsule layer: a Conv2D projects a spatial feature map into
    (num_capsules * capsule_dim) channels, then reshapes each spatial location's
    channel group into one capsule vector, and squashes it."""

    def __init__(self, num_capsules: int, capsule_dim: int, kernel_size: int = 3,
                 strides: int = 2, **kwargs):
        super().__init__(**kwargs)
        self.num_capsules = num_capsules
        self.capsule_dim = capsule_dim
        self.conv = layers.Conv2D(
            num_capsules * capsule_dim, kernel_size, strides=strides,
            padding="same", activation=None, name="primary_caps_conv",
        )

    def call(self, inputs: tf.Tensor) -> tf.Tensor:
        """inputs: (batch, H, W, C) spatial feature map.
        Returns: (batch, n_primary_caps, capsule_dim) - flattened over space and
        the num_capsules groups, each a squashed vector."""
        x = self.conv(inputs)  # (batch, h, w, num_capsules * capsule_dim)
        batch = tf.shape(x)[0]
        h, w = x.shape[1], x.shape[2]
        n_primary_caps = h * w * self.num_capsules
        x = tf.reshape(x, (batch, n_primary_caps, self.capsule_dim))
        return squash(x, axis=-1)


class RoutingCaps(layers.Layer):
    """Routing-by-agreement between a PrimaryCaps layer and this layer's output
    capsules (one per class). Each input capsule casts a weighted "vote" for each
    output capsule (via a learned transform W); routing coefficients are refined
    over `n_routing` iterations based on how well each vote agrees with the
    current output capsule (dot-product agreement), per Sabour et al.'s
    Algorithm 1 - this is what "routing-by-agreement" in the architecture
    diagram refers to, not a fixed pooling/summation."""

    def __init__(self, num_output_capsules: int, output_capsule_dim: int,
                 n_routing: int = 3, **kwargs):
        super().__init__(**kwargs)
        self.num_output_capsules = num_output_capsules
        self.output_capsule_dim = output_capsule_dim
        self.n_routing = n_routing

    def build(self, input_shape):
        n_input_caps, input_capsule_dim = input_shape[1], input_shape[2]
        # W: one learned transform matrix per (input capsule, output capsule) pair
        self.W = self.add_weight(
            name="routing_weights",
            shape=(n_input_caps, self.num_output_capsules, self.output_capsule_dim, input_capsule_dim),
            initializer="glorot_uniform",
            trainable=True,
        )

    def call(self, inputs: tf.Tensor) -> tf.Tensor:
        """inputs: (batch, n_input_caps, input_capsule_dim)
        Returns: (batch, num_output_capsules, output_capsule_dim)"""
        batch = tf.shape(inputs)[0]
        n_input_caps = inputs.shape[1]

        # votes[b,i,j,:] = W[i,j] @ inputs[b,i,:]  -> (batch, n_in, n_out, out_dim).
        # einsum broadcasts the batch dim automatically (W has none) - no manual
        # tiling needed, which is exactly where the shape bug above came from.
        votes = tf.einsum("ijod,bid->bijo", self.W, inputs)

        routing_logits = tf.zeros((batch, n_input_caps, self.num_output_capsules))
        votes_stopped = tf.stop_gradient(votes)

        outputs = None
        for i in range(self.n_routing):
            routing_weights = tf.nn.softmax(routing_logits, axis=2)  # agreement coefficients c_ij, sum over j=1
            routing_weights = routing_weights[..., tf.newaxis]        # (b, n_in, n_out, 1)

            use_votes = votes if i == self.n_routing - 1 else votes_stopped
            weighted_sum = tf.reduce_sum(routing_weights * use_votes, axis=1)  # (b, n_out, out_dim)
            outputs = squash(weighted_sum, axis=-1)

            if i < self.n_routing - 1:
                agreement = tf.reduce_sum(votes_stopped * outputs[:, tf.newaxis, :, :], axis=-1)  # (b, n_in, n_out)
                routing_logits = routing_logits + agreement

        return outputs


def margin_loss(labels: tf.Tensor, lengths: tf.Tensor, m_plus: float = 0.9,
                m_minus: float = 0.1, down_weight: float = 0.5) -> tf.Tensor:
    """The margin loss from Sabour et al. 2017, eq. 4 - the loss capsule lengths were designed for.
    labels: (batch,) integer class ids; lengths: (batch, n_classes) in [0, 1).

    Why not softmax cross-entropy on the lengths: lengths are bounded in [0, 1), so a softmax over
    them can never exceed ~e/(e+3) = 0.475 for one of 4 classes - the loss floors near 0.74 and the
    gradient signal is weak (measured locally: predictions collapsed to one class). The margin loss
    instead pushes the true class's length above m_plus and every other class's below m_minus."""
    targets = tf.one_hot(labels, depth=lengths.shape[-1], dtype=lengths.dtype)
    present = targets * tf.square(tf.nn.relu(m_plus - lengths))
    absent = down_weight * (1.0 - targets) * tf.square(tf.nn.relu(lengths - m_minus))
    return tf.reduce_mean(tf.reduce_sum(present + absent, axis=-1))


def capsule_lengths(output_capsules: tf.Tensor, epsilon: float = 1e-7) -> tf.Tensor:
    """(batch, n_classes, dim) -> (batch, n_classes) vector lengths, read as
    per-class confidence. Softmax over these gives a proper severity probability
    distribution for use with standard categorical cross-entropy."""
    return tf.sqrt(tf.reduce_sum(tf.square(output_capsules), axis=-1) + epsilon)


In [ ]:
%%writefile model.py
"""
CNN -> LSTM encoder-decoder for spatiotemporal flood depth forecasting.

Input: (batch, T_in, H, W, C=3) - depth[t], static DEM, rainfall[t] per historical timestep (rainfall = intensity in mm/h of the interval containing t: partly future for the last frame, the 'hybrid' input).
Output: (batch, T_out, H, W) depth regression only.

This is the depth-forecasting branch only, per the current (Capsule Network + Symbolic
Rule Layer) architecture - the per-pixel dry/flooded classifier this module used to carry
predates that pivot and has been removed. Severity classification now lives entirely in
the separate Capsule Network branch (build_capsule_head, added in Stage 3), which reads
the same shared CNN-encoder features but is trained/evaluated independently of this module.

Architecture follows the dossier's Section 4.5 illustrative design (time-distributed CNN
encoder -> LSTM temporal encoder-decoder -> CNN decoder), itself following the
time-distributed-CNN-then-LSTM pattern from Malik et al. 2024 and the CNN/LSTM-fused-at-
bottleneck pattern from Wang et al. 2026 (see research dossier, Section 2.B).

CO mapping:
  CO1 - dense_bottleneck: Dense layer projecting CNN features to the LSTM's input vector
  CO2 - severity classification head: now the Capsule Network (see Stage 3), not this module
  CO3 - cnn_encoder_*/cnn_decoder_*: convolutional spatial feature extraction/reconstruction
  CO4 - dropout/optimizer/learning_rate exposed as build_models() parameters
  CO5 - encoder_lstm/decoder_lstm: the temporal encoder-decoder generating the forecast sequence
        (round 2: in the trained ORIGINAL model (v1) the depth history was not exploited - see PROJECT_LOG 'Completion, round 2'; the final ConvLSTM is model_v2.py)
"""

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from capsule_layers import PrimaryCaps, RoutingCaps, capsule_lengths

PATCH_SIZE = 128  # H = W = 128, per dossier Section 4.5
N_CHANNELS = 3    # depth, DEM, rainfall
NUM_SEVERITY_CLASSES = 4  # none / minor / moderate / severe, see real_data_loader.py

# Fixed (non-learned) scales so raw physical inputs feed the CNN sanely. Without these,
# DEM values (tens to hundreds of metres) swamp rainfall (0-9 mm/h) and depth (0-18 m).
DEPTH_SCALE_M = 5.0     # input depth is divided by this; the decoder multiplies back to metres
DEM_SCALE_M = 50.0      # DEM is made patch-relative (minus its own mean) then divided by this
RAIN_SCALE_MM = 5.0     # in mm/h (the rainfall files hold intensities)
DELTA_SCALE_M = 0.05    # the decoder's unit of "change from the last observed frame" is 5 cm


class InputNormalizer(layers.Layer):
    """Fixed per-channel scaling applied INSIDE the model, so every caller (training,
    predict_severity, Grad-CAM) keeps passing raw physical units and getting metres back.

    The DEM channel is patch-relative (mean removed) rather than absolute elevation: absolute
    elevation is a region-specific fingerprint that can't transfer to a new region, while
    relative relief (slopes, valleys) can - which matters for the cross-regional test."""

    def call(self, x):  # x: (batch, T, H, W, 3) raw units
        depth = x[..., 0:1] / DEPTH_SCALE_M
        dem = x[..., 1:2]
        dem_rel = (dem - tf.reduce_mean(dem, axis=(2, 3), keepdims=True)) / DEM_SCALE_M
        rain = x[..., 2:3] / RAIN_SCALE_MM
        return tf.concat([depth, dem_rel, rain], axis=-1)

OPTIMIZERS = {
    "adam": keras.optimizers.Adam,
    "rmsprop": keras.optimizers.RMSprop,
    "sgd": keras.optimizers.SGD,
}


def build_encoder(hidden_dim: int = 128, dropout: float = 0.3) -> keras.Model:
    """Shared CNN encoder + LSTM temporal encoder. Outputs both the LSTM's final
    states (for the depth-forecasting decoder) AND the raw per-timestep CNN
    feature maps, BEFORE they're flattened into the LSTM - this second output is
    what the Capsule Network (Stage 3) consumes, so both branches genuinely share
    one encoder and both receive real gradients through it, rather than the
    Capsule Network operating on a separately re-computed copy."""
    inputs = keras.Input(shape=(None, PATCH_SIZE, PATCH_SIZE, N_CHANNELS), name="input_sequence")

    x = InputNormalizer(name="input_normalizer")(inputs)
    x = layers.TimeDistributed(layers.Conv2D(16, 3, strides=2, padding="same", activation="relu"), name="cnn_encoder_1")(x)
    x = layers.TimeDistributed(layers.Conv2D(32, 3, strides=2, padding="same", activation="relu"), name="cnn_encoder_2")(x)
    cnn_features = layers.TimeDistributed(layers.Conv2D(64, 3, strides=2, padding="same", activation="relu"), name="cnn_encoder_3")(x)
    # 128x128 -> 16x16x64 after three stride-2 blocks; cnn_features: (batch, T, 16, 16, 64)

    x = layers.TimeDistributed(layers.Flatten())(cnn_features)
    x = layers.TimeDistributed(layers.Dropout(dropout))(x)
    x = layers.TimeDistributed(layers.Dense(hidden_dim, activation="relu"), name="dense_bottleneck")(x)  # CO1

    _outputs, state_h, state_c = layers.LSTM(
        hidden_dim, return_sequences=True, return_state=True, dropout=dropout, name="encoder_lstm"
    )(x)

    return keras.Model(inputs, [state_h, state_c, cnn_features], name="encoder")


def build_decoder_step(hidden_dim: int = 128, dropout: float = 0.3) -> keras.Model:
    """One decode step. The LSTM takes a constant learned "step" input and relies entirely
    on its (state_h, state_c) carried from the previous step to produce a different output
    each time - standard for this kind of generative decoder, and much cheaper than
    re-encoding the full previous frame through a dense layer just to feed it back in.

    PERSISTENCE-ANCHORED (residual): the output is the most recent OBSERVED depth frame plus
    a learned change, not a depth field rebuilt from scratch. The first version rebuilt the
    whole 128x128 field from a 128-number LSTM state with no link to the observed frame, so
    it could never approach a near-exact copy - and a copy of the last frame is already
    accurate to ~2 mm at a 5-minute step, ~13 mm at 30 minutes (measured on real data).
    That, not only "too little data", is why the early predictions were blurry. The final
    conv is zero-initialized, so training starts exactly at the persistence forecast and
    can only improve on it."""
    step_input = keras.Input(shape=(1,), name="step_input")  # constant scalar, e.g. tf.ones((1,1))
    state_h_in = keras.Input(shape=(hidden_dim,), name="state_h")
    state_c_in = keras.Input(shape=(hidden_dim,), name="state_c")
    anchor_in = keras.Input(shape=(PATCH_SIZE, PATCH_SIZE, 1), name="anchor_depth")  # last observed depth, metres
    skip_in = keras.Input(shape=(16, 16, 64), name="skip_features")  # encoder's last-step spatial features

    step_emb = layers.Dense(hidden_dim, name="step_embedding")(step_input)
    step_emb_seq = layers.Reshape((1, hidden_dim))(step_emb)
    lstm_out, state_h, state_c = layers.LSTM(hidden_dim, return_state=True, name="decoder_lstm")(
        step_emb_seq, initial_state=[state_h_in, state_c_in]
    )
    lstm_out = layers.Dropout(dropout)(lstm_out)

    # project back to a 16x16x64 feature map, then upsample with transposed convs (mirrors the encoder)
    x = layers.Dense(16 * 16 * 64, activation="relu", name="decoder_bottleneck")(lstm_out)
    x = layers.Reshape((16, 16, 64))(x)
    # Skip connection: the LSTM state alone is a 128-number summary, so the decoder can express a
    # global drift but has no way to say WHERE anything changes. Concatenating the encoder's own
    # last-step spatial features gives every 16x16 cell access to the local observed structure.
    # (Measured: without this, held-out skill plateaued at exactly +2% from epoch 2 onward, and
    # was the same +2% on a region never seen in training - a constant drift, not learning.)
    x = layers.Concatenate(name="context_plus_skip")([x, skip_in])
    x = layers.Conv2DTranspose(32, 3, strides=2, padding="same", activation="relu", name="cnn_decoder_1")(x)
    x = layers.Conv2DTranspose(16, 3, strides=2, padding="same", activation="relu", name="cnn_decoder_2")(x)
    x = layers.Conv2DTranspose(16, 3, strides=2, padding="same", activation="relu", name="cnn_decoder_3")(x)
    # 16x16 -> 32x32 -> 64x64 -> 128x128

    delta = layers.Conv2D(1, 1, activation="linear", kernel_initializer="zeros", name="depth_delta")(x)
    delta = layers.Rescaling(DELTA_SCALE_M, name="delta_to_meters")(delta)
    depth_out = layers.Add(name="anchor_plus_delta")([anchor_in, delta])               # (batch,128,128,1) metres

    return keras.Model(
        [step_input, state_h_in, state_c_in, anchor_in, skip_in],
        [depth_out, state_h, state_c],
        name="decoder_step",
    )


def anchor_from_input(x):
    """Most recent observed depth frame from a raw (batch, T, H, W, 3) input -> (batch, H, W, 1)."""
    return x[:, -1, :, :, 0:1]


def decode_horizon(decoder, enc_out, x, horizon: int, training: bool = False):
    """Runs the residual decoder for `horizon` steps. Returns (batch, horizon, H, W) depth in metres.

    enc_out: the encoder's outputs [state_h, state_c, cnn_features]; x: the SAME raw input batch
    the encoder was run on (its last depth frame is the persistence anchor). Every step is
    anchored to that frame and sees the encoder's last-step spatial features; the LSTM state
    evolves the change."""
    state_h, state_c, cnn_features = enc_out
    anchor_depth = anchor_from_input(x)
    skip = cnn_features[:, -1]
    step_input = tf.ones((tf.shape(state_h)[0], 1))
    frames = []
    sh, sc = state_h, state_c
    for _ in range(horizon):
        depth_out, sh, sc = decoder([step_input, sh, sc, anchor_depth, skip], training=training)
        frames.append(depth_out[..., 0])
    return tf.stack(frames, axis=1)


def build_capsule_head(num_primary_capsules: int = 8, primary_capsule_dim: int = 8,
                        severity_capsule_dim: int = 16, n_routing: int = 3,
                        return_lengths: bool = False) -> keras.Model:
    """Severity-classification branch (Stage 3): PrimaryCaps -> routing-by-agreement
    -> one capsule per severity class. Consumes the LAST timestep of the SAME
    cnn_features the encoder outputs - i.e. the shared encoder's own most-recent
    observed spatial representation, not a separately-computed copy - so this
    branch's gradients flow back into the real shared encoder weights.

    return_lengths=False (default): output softmax(lengths), for use with cross-entropy.
    return_lengths=True: output the raw capsule lengths in [0, 1) for use with
    capsule_layers.margin_loss (the loss the paper designed lengths for). argmax is the
    prediction either way."""
    cnn_feat_input = keras.Input(shape=(16, 16, 64), name="cnn_features_last_step")
    primary = PrimaryCaps(num_primary_capsules, primary_capsule_dim, kernel_size=3, strides=2)(cnn_feat_input)
    severity_caps = RoutingCaps(NUM_SEVERITY_CLASSES, severity_capsule_dim, n_routing=n_routing)(primary)
    lengths = layers.Lambda(capsule_lengths, name="capsule_lengths")(severity_caps)
    if return_lengths:
        return keras.Model(cnn_feat_input, lengths, name="capsule_severity_head")
    probs = layers.Softmax(name="severity_probs")(lengths)
    return keras.Model(cnn_feat_input, probs, name="capsule_severity_head")


def build_models(hidden_dim: int = 128, dropout: float = 0.3):
    return build_encoder(hidden_dim, dropout), build_decoder_step(hidden_dim, dropout)


def forecast(encoder, decoder, input_sequence: np.ndarray, horizon: int, hidden_dim: int = 128):
    """Real forward pass: encode T_in history, then decode `horizon` future frames."""
    x = tf.constant(input_sequence[np.newaxis, ...].astype("float32"))  # (1, T_in, H, W, C)
    enc_out = encoder(x, training=False)

    shapes = {"input": tuple(x.shape), "encoder_state_h": tuple(enc_out[0].shape)}
    frames = decode_horizon(decoder, enc_out, x, horizon, training=False)
    shapes["decoded_frames"] = tuple(frames.shape)
    return frames.numpy()[0], shapes


def dummy_training_step(encoder, decoder, input_sequence: np.ndarray, horizon: int,
                         optimizer_name: str = "adam", learning_rate: float = 1e-3, hidden_dim: int = 128):
    """One real gradient step against random fake targets - proves the loss/optimizer/dropout
    wiring (CO4) is mechanically correct. Not real training; the targets are random."""
    optimizer = OPTIMIZERS[optimizer_name](learning_rate=learning_rate)
    x = tf.constant(input_sequence[np.newaxis, ...].astype("float32"))
    rng = np.random.default_rng(0)
    fake_depth_targets = tf.constant(rng.random((horizon, PATCH_SIZE, PATCH_SIZE)).astype("float32"))

    trainable_vars = encoder.trainable_variables + decoder.trainable_variables

    with tf.GradientTape() as tape:
        enc_out = encoder(x, training=True)
        frames = decode_horizon(decoder, enc_out, x, horizon, training=True)
        loss = tf.reduce_mean(tf.square(frames[0] - fake_depth_targets))

    grads = tape.gradient(loss, trainable_vars)
    n_none = sum(1 for g in grads if g is None)
    optimizer.apply_gradients((g, v) for g, v in zip(grads, trainable_vars) if g is not None)

    return float(loss.numpy()), n_none, len(trainable_vars)


In [ ]:
%%writefile grad_cam.py
"""
Stage 4: Grad-CAM (Selvaraju et al. 2017) for the depth-forecasting branch,
adapted from classification to dense regression - there's no single "class
score" to explain, so the target is the SUM of predicted depth over the output
patch at a chosen forecast step, a scalar standing in for "how much flood did
the model predict, in total." Explains cnn_encoder_3 (the encoder's last, most
semantically-rich conv layer - standard Grad-CAM practice), at the input
sequence's LAST timestep, matching how the capsule branch also reads the
encoder's most recent observed state.

Purely post-hoc: no retraining, gradients here are for the heatmap only and are
never applied to any weights.
"""
import numpy as np
import tensorflow as tf

import model as flood_model


def grad_cam(encoder, decoder, input_sequence: np.ndarray, horizon_step: int = 0):
    """input_sequence: (T_in, H, W, 3). Returns (heatmap, net_change_m): a (H, W) heatmap in
    [0,1], upsampled from the encoder's 16x16 feature-map resolution to the full input
    resolution, and the model's predicted net change in depth (metres, summed over the patch)
    at `horizon_step`. The sign of net_change_m says which direction the map explains:
    positive = evidence for the predicted rise, negative = evidence for the predicted fall."""
    x_batch = tf.constant(input_sequence[np.newaxis, ...], dtype=tf.float32)

    with tf.GradientTape() as tape:
        enc_out = encoder(x_batch, training=False)
        cnn_features = enc_out[2]
        tape.watch(cnn_features)  # the same tensor the decoder's skip connection reads, so both paths count

        frames = flood_model.decode_horizon(decoder, enc_out, x_batch, horizon_step + 1, training=False)
        # The decoder is persistence-anchored, so the gradient of the predicted depth w.r.t. the
        # encoder features reflects what drives the model's predicted CHANGE from the last
        # observed frame (the anchor itself carries no gradient to the features).
        #
        # Direction matters. Grad-CAM's ReLU keeps only evidence that INCREASES the target, so
        # if the model predicts net drying the plain map is entirely negative and ReLU blanks
        # it (measured on a real validation window: net change -0.11 m, 0% of cells positive).
        # So the target is signed by the predicted net change - decided outside the gradient
        # path - and the map shows evidence for the direction the model actually predicts.
        predicted_depth = tf.reduce_sum(frames[0, horizon_step])
        net_change = predicted_depth - tf.reduce_sum(flood_model.anchor_from_input(x_batch))
        direction = tf.stop_gradient(tf.where(net_change >= 0, 1.0, -1.0))
        target = direction * predicted_depth

    grads = tape.gradient(target, cnn_features)  # (1, T_in, 16, 16, 64)
    last_step_grads = grads[:, -1]                # (1, 16, 16, 64) - attribute to the most recent step
    last_step_features = cnn_features[:, -1]

    channel_weights = tf.reduce_mean(last_step_grads, axis=(0, 1, 2))  # (64,) global-avg-pool, per Grad-CAM
    weighted = tf.reduce_sum(last_step_features[0] * channel_weights, axis=-1)  # (16, 16)
    heatmap = tf.nn.relu(weighted)

    heatmap = heatmap / (tf.reduce_max(heatmap) + 1e-9)  # normalize to [0,1]
    heatmap_full_res = tf.image.resize(
        heatmap[tf.newaxis, ..., tf.newaxis], (flood_model.PATCH_SIZE, flood_model.PATCH_SIZE), method="bilinear"
    )[0, ..., 0]

    return heatmap_full_res.numpy(), float(net_change)


In [ ]:
%%writefile symbolic_rule_layer.py
"""
Stage 5: Symbolic Rule Layer (NeuroSymbolic) - post-hoc, deterministic rule logic
applied AFTER the trained model produces its outputs. No weights, no training,
no gradients through this module - it is a plain function, which is exactly
what makes the "NeuroSymbolic" label accurate (see project explainer doc).

All four thresholds below were derived from real Australia (30m, high-fidelity)
data spanning all 4 severity classes (10 examples each, from
train_stage3_balanced.py's class-balanced scan), NOT guessed:

  depth-rate (patch-mean, m/hr):  none=0.0000  minor=0.0002  moderate=0.0028  severe=0.0035
  rainfall (patch-mean, mm/h):    none=0.000   minor=0.189   moderate=2.055   severe=4.056
  slope (% grade):                none=4.85    minor=4.53    moderate=5.79    severe=8.22
  drainage-proximity (m to       none=911.9   minor=774.4   moderate=1030.1  severe=1260.6
  nearest baseline water px):

Two of these ran counter to the naive assumption before checking real data:
higher drainage-proximity and higher slope both correlate with MORE severe
outcomes here, not less. The physical reading: severe flooding means water
reaching the floodplain, farther from the baseline channel than routine minor
flooding - and often on higher-energy, steeper terrain, not just pooling in
flat low spots. The rules below follow the real data's direction, not the
initial (wrong) assumption.
"""
from real_data_loader import SEVERITY_CLASSES

DEPTH_RATE_ESCALATE_THRESHOLD = 0.0030   # m/hr, patch-mean - near the moderate/severe boundary
DEPTH_RATE_RECEDE_THRESHOLD = -0.0001    # m/hr, patch-mean - mildly negative = receding
RAINFALL_CORROBORATE_THRESHOLD = 3.0     # mm/h, patch-mean - near the moderate/severe boundary
SLOPE_HIGH_ENERGY_THRESHOLD = 7.0        # % grade - near the moderate/severe boundary
DRAINAGE_FLOODPLAIN_THRESHOLD = 1000.0   # meters - near the moderate/severe boundary


def apply_symbolic_rules(raw_severity_idx: int, depth_rate_m_per_hr: float, rainfall_mm: float,
                          slope_pct: float, drainage_proximity_m: float):
    """Returns (reasoned_severity_idx, list_of_justification_strings).
    raw_severity_idx: the Capsule Network's own prediction (0=none..3=severe) - the
    starting point these rules adjust, not replace outright."""
    severity_idx = raw_severity_idx
    reasons = []

    if depth_rate_m_per_hr > DEPTH_RATE_ESCALATE_THRESHOLD and drainage_proximity_m > DRAINAGE_FLOODPLAIN_THRESHOLD:
        if severity_idx < len(SEVERITY_CLASSES) - 1:
            severity_idx += 1
        reasons.append(
            f"escalated: depth rising {depth_rate_m_per_hr:.4f} m/hr while {drainage_proximity_m:.0f} m "
            f"from the baseline water extent - a floodplain-inundation pattern, not just channel rise"
        )

    if depth_rate_m_per_hr < DEPTH_RATE_RECEDE_THRESHOLD:
        if severity_idx > 0:
            severity_idx -= 1
        reasons.append(f"de-escalated: depth trend is receding ({depth_rate_m_per_hr:.4f} m/hr)")

    if rainfall_mm > RAINFALL_CORROBORATE_THRESHOLD and depth_rate_m_per_hr > 0:
        reasons.append(
            f"rainfall intensity ({rainfall_mm:.2f} mm/h, current 30-minute interval) accompanies the rising depth trend"
        )

    if slope_pct > SLOPE_HIGH_ENERGY_THRESHOLD and depth_rate_m_per_hr > 0:
        reasons.append(
            f"steep terrain ({slope_pct:.1f}% grade) suggests high-energy flow, not simple pooling"
        )

    if not reasons:
        reasons.append("no rule triggered an adjustment; raw model classification retained")

    return severity_idx, reasons


In [ ]:
%%writefile predict_severity.py
"""
Stage 5b: predict_severity_with_reasoning() - the actual usable inference path,
combining the trained Capsule Network (Stage 3) with the Symbolic Rule Layer
(Stage 5) into one call. This is what a demo, or Stage 6's evaluation, calls -
not the individual pieces tested in isolation.
"""
import warnings

import numpy as np
from scipy import ndimage
import tensorflow as tf

from model import decode_horizon
from real_data_loader import RegionData, SEVERITY_CLASSES
from symbolic_rule_layer import apply_symbolic_rules

# The rule thresholds in symbolic_rule_layer.py were derived from Australia at 30m only.
RULE_CALIBRATION_PIXEL_SIZE_M = 30.0


def _compute_slope_pct(dem_patch: np.ndarray, pixel_size_m: float) -> float:
    gy, gx = np.gradient(dem_patch, pixel_size_m)
    return float(100 * np.mean(np.sqrt(gx ** 2 + gy ** 2)))


def _compute_drainage_proximity_m(landuse_patch: np.ndarray, pixel_size_m: float) -> float:
    water_mask = (landuse_patch == 1)  # verified against the real Esri/Impact Observatory legend
    if not water_mask.any():
        return float("nan")  # no baseline water anywhere in this patch - proximity undefined, not zero
    dist_px = ndimage.distance_transform_edt(~water_mask)
    return float(np.mean(dist_px) * pixel_size_m)


def predict_severity_with_reasoning(encoder, decoder, capsule_head, rd: RegionData,
                                     row: int, col: int, patch_size: int, start_idx: int,
                                     t_in: int, horizon: int, stride: int = 1):
    """Runs the real forward pass + rule layer on one real patch/time window.
    Returns a dict with the raw capsule prediction, the reasoned output, the
    justification, and the underlying feature values (for transparency/debugging).
    `stride` is the window's time stride in depth steps (6 = 30 min); depth-rate is
    computed per hour of real time, so it stays comparable across strides."""
    if rd.pixel_size_m != RULE_CALIBRATION_PIXEL_SIZE_M:
        warnings.warn(
            f"{rd.region} is at {rd.pixel_size_m:.0f}m but symbolic_rule_layer.py's thresholds were "
            f"derived at {RULE_CALIBRATION_PIXEL_SIZE_M:.0f}m (Australia) - a 128px patch covers a very "
            f"different physical area, so the rules are not calibrated for this region."
        )

    input_seq, depth_targets, true_severity, max_future_depth = rd.load_patch_sequence(
        row, col, patch_size, start_idx, t_in, horizon, stride
    )

    x_batch = tf.constant(input_seq[np.newaxis, ...], dtype=tf.float32)
    enc_out = encoder(x_batch, training=False)

    severity_scores = capsule_head(enc_out[2][:, -1], training=False)  # probs or lengths; argmax is the same
    raw_severity_idx = int(tf.argmax(severity_scores[0]))

    pred_depth_frames = decode_horizon(
        decoder, enc_out, x_batch, horizon, training=False
    ).numpy()[0]  # (horizon, H, W) - model's OWN depth prediction,
    # used for depth-rate, not the ground-truth depth_targets - this is what the reasoning layer
    # would actually have access to at real inference time (no ground truth available then)

    # Depth-rate = the model's PREDICTED change in patch-mean depth, from the last observed frame
    # to the last forecast frame, per hour of real time. (An earlier version differenced the
    # first and last FORECAST frames. With the original from-scratch decoder those frames were
    # essentially noise, so the rate that triggered the escalation rule was an artifact; with the
    # persistence-anchored decoder that difference is ~0 by construction. Measuring from the
    # observed frame uses the observation plus the model's predicted trend, which is the quantity
    # the rule thresholds were derived from - the real patch-mean change per hour.)
    last_observed_mean = float(input_seq[-1, ..., 0].mean())
    hours = horizon * stride * rd.depth_step_seconds / 3600.0
    depth_rate_m_per_hr = float((pred_depth_frames[-1].mean() - last_observed_mean) / hours)

    rainfall_mm = float(input_seq[-1, ..., 2].mean())   # mm/h; the hybrid input includes rain of the interval containing the last frame (partly not yet fallen)

    dem_patch = rd.load_dem_patch(row, col, patch_size)
    slope_pct = _compute_slope_pct(dem_patch, rd.pixel_size_m)

    landuse_patch = rd.load_landuse_patch(row, col, patch_size)
    drainage_proximity_m = _compute_drainage_proximity_m(landuse_patch, rd.pixel_size_m)

    reasoned_idx, reasons = apply_symbolic_rules(
        raw_severity_idx, depth_rate_m_per_hr, rainfall_mm, slope_pct,
        drainage_proximity_m if not np.isnan(drainage_proximity_m) else 0.0,
    )

    return {
        "true_severity": SEVERITY_CLASSES[true_severity],
        "raw_severity": SEVERITY_CLASSES[raw_severity_idx],
        "reasoned_severity": SEVERITY_CLASSES[reasoned_idx],
        "justification": "; ".join(reasons),
        "features": {
            "depth_rate_m_per_hr": depth_rate_m_per_hr,
            "rainfall_mm": rainfall_mm,
            "slope_pct": slope_pct,
            "drainage_proximity_m": drainage_proximity_m,
        },
        "pred_depth_frames": pred_depth_frames,
        "depth_targets": depth_targets,
    }


In [ ]:
%%writefile windows.py
"""
Window scanning and sampling for FloodCastBench regions.

A "window" is one (spatial patch, start time). Severity is computed EXACTLY the way
RegionData.load_patch_sequence computes it - max depth in the patch over the future
frames from RegionData.window_frame_indices - and both use that same method, so a scan
can never disagree with what the loader later returns.

Each scanned window is a tuple:
    (row, col, start_idx, severity, baseline_severity)
where baseline_severity is what you'd get by just applying the severity thresholds to the
max depth of the LAST OBSERVED frame (a "persistence" classifier that needs no model).
Measured on real Australia data, that trivial classifier is 99.9% accurate at a 5-minute
step and 96.7% at a 30-minute stride - so any severity model has to be judged against it.
"""
import numpy as np
import rasterio

from real_data_loader import SEVERITY_CLASSES, RegionData, depth_to_severity, has_nodata


def all_positions(rd: RegionData, patch_size: int):
    """Non-overlapping patch origins that fit entirely inside the depth grid."""
    return [(r, c) for r in range(0, rd.grid_height - patch_size + 1, patch_size)
                   for c in range(0, rd.grid_width - patch_size + 1, patch_size)]


def valid_positions(rd: RegionData, patch_size: int, dem_nodata_std: float = 1.0):
    """Drop patches whose DEM is (near-)flat - those lie outside the simulation domain - and patches whose
    DEM window touches the NoData sentinel (its std is enormous, so the flat-DEM test alone would keep it)."""
    keep = []
    for (r, c) in all_positions(rd, patch_size):
        dem = rd.load_dem_patch(r, c, patch_size)
        if not has_nodata(dem) and dem.std() > dem_nodata_std:
            keep.append((r, c))
    return keep


def start_indices_for(rd: RegionData, count: int, t_in: int, horizon: int, stride: int):
    """`count` start indices spread evenly over every start at which a whole window fits."""
    last = rd.max_start_index(t_in, horizon, stride)
    return np.unique(np.linspace(0, last, count).astype(int))


def scan_windows(rd: RegionData, positions, start_indices, patch_size: int,
                 t_in: int, horizon: int, stride: int):
    out = []
    for start_idx in start_indices:
        history_idx, future_idx = rd.window_frame_indices(int(start_idx), t_in, horizon, stride)
        with rasterio.open(rd.depth_files[history_idx[-1]]) as src:
            last_observed = src.read(1)
        future_max = None
        for i in future_idx:
            with rasterio.open(rd.depth_files[i]) as src:
                frame = src.read(1)
            future_max = frame if future_max is None else np.maximum(future_max, frame)
        for (row, col) in positions:
            sl = (slice(row, row + patch_size), slice(col, col + patch_size))
            out.append((row, col, int(start_idx),
                        depth_to_severity(float(future_max[sl].max())),
                        depth_to_severity(float(last_observed[sl].max()))))
    return out


def class_fractions(windows):
    total = len(windows)
    return {c: sum(1 for w in windows if w[3] == i) / total for i, c in enumerate(SEVERITY_CLASSES)}


def balanced_sample(pool, per_class: int, rng):
    """Up to `per_class` randomly-chosen windows of each severity class."""
    chosen = []
    for cls_idx, cls in enumerate(SEVERITY_CLASSES):
        members = [w for w in pool if w[3] == cls_idx]
        take = min(per_class, len(members))
        if take < per_class:
            print(f"  WARNING: only {len(members)} '{cls}' windows available (wanted {per_class})")
        pick = rng.choice(len(members), size=take, replace=False)
        chosen.extend(members[i] for i in pick)
    return chosen


## Load the two usable regions

In [ ]:
import sys
sys.path.insert(0, ".")
import numpy as np
from real_data_loader import RegionData, SEVERITY_CLASSES, depth_to_severity

REGIONS = {
    "Australia": "High-fidelity flood forecasting/30m",
    "UK": "High-fidelity flood forecasting/30m",
}
regions = {name: RegionData(root_dir=DATA_ROOT, region=name, fidelity_dir=fid) for name, fid in REGIONS.items()}
for name, rd in regions.items():
    print(f"{name}: {len(rd.depth_files)} depth frames, grid {rd.grid_height}x{rd.grid_width}, "
          f"{rd.pixel_size_m:.0f}m, layer offset {rd.layer_offset}")

print("(Pakistan, Mozambique and the 60m data are not downloaded here - see flood_forecasting_multires_colab.ipynb.)")

## Window pool and severity-class census

A window is one (spatial patch, start time). Severity is computed exactly as the loader computes it
(max depth in the patch over the forecast frames). The census reproduces the pre-flight finding.

In [ ]:
import rasterio
from windows import (valid_positions, start_indices_for, scan_windows, class_fractions, balanced_sample)

print("Severity-class census (fraction of windows in each class):")
for name, rd in regions.items():
    positions = valid_positions(rd, PATCH_SIZE, DEM_NODATA_STD)
    starts = list(range(0, rd.max_start_index(T_IN, HORIZON, TIME_STRIDE) + 1, CENSUS_START_STEP))
    fr = class_fractions(scan_windows(rd, positions, starts, PATCH_SIZE, T_IN, HORIZON, TIME_STRIDE))
    print(f"  {name:10s} patches={len(positions):2d}  " + "  ".join(f"{c}={100 * fr[c]:.0f}%" for c in SEVERITY_CLASSES))
print("-> Only Australia has real class diversity. Severity is trained and validated there only.")

## Australia: split by spatial position, then class-balance within each side

Validation patches are **never seen in training** - not even at other times.

In [ ]:
rd_au = regions["Australia"]
rng = np.random.default_rng(SEED)

au_positions = valid_positions(rd_au, PATCH_SIZE, DEM_NODATA_STD)
perm = rng.permutation(len(au_positions))
n_val_pos = max(1, int(round(len(au_positions) * VAL_POSITION_FRACTION)))
val_positions = [au_positions[i] for i in perm[:n_val_pos]]
train_positions = [au_positions[i] for i in perm[n_val_pos:]]
print(f"Australia patches: {len(au_positions)} valid -> {len(train_positions)} train / {len(val_positions)} val")
assert not set(train_positions) & set(val_positions)

starts = start_indices_for(rd_au, N_START_INDICES, T_IN, HORIZON, TIME_STRIDE)
print(f"scanning {len(starts)} start times x {len(au_positions)} patches ...")
train_pool = scan_windows(rd_au, train_positions, starts, PATCH_SIZE, T_IN, HORIZON, TIME_STRIDE)
val_pool = scan_windows(rd_au, val_positions, starts, PATCH_SIZE, T_IN, HORIZON, TIME_STRIDE)

train_windows = balanced_sample(train_pool, PER_CLASS_TRAIN, rng)
val_windows = balanced_sample(val_pool, PER_CLASS_VAL, rng)
uniform_idx = rng.choice(len(val_pool), size=min(N_UNIFORM_AU, len(val_pool)), replace=False)
au_uniform_windows = [val_pool[i] for i in uniform_idx]
print(f"train={len(train_windows)}  val(balanced)={len(val_windows)}  val(uniform, for UK comparison)={len(au_uniform_windows)}")

## Load window arrays

In [ ]:
def load_examples(rd, windows):
    examples = []
    for (row, col, start_idx, expected_sev, _baseline) in windows:
        input_seq, depth_targets, severity, _ = rd.load_patch_sequence(
            row, col, PATCH_SIZE, start_idx, T_IN, HORIZON, TIME_STRIDE
        )
        assert severity == expected_sev, "scan and loader disagree on severity - they must use the same rule"
        examples.append((input_seq, depth_targets, severity))
    return examples


def shuffled(items, seed):
    order = np.random.default_rng(seed).permutation(len(items))
    return [items[i] for i in order]


train_windows = shuffled(train_windows, 1)
val_windows = shuffled(val_windows, 2)

print("loading training windows ...")
train_data = load_examples(rd_au, train_windows)
print("loading validation windows ...")
val_data = load_examples(rd_au, val_windows)
au_uniform_data = load_examples(rd_au, au_uniform_windows)
print(f"train {len(train_data)}  val {len(val_data)}  au_uniform {len(au_uniform_data)}")

## The baselines everything is judged against

- **Depth - persistence:** predict that every future frame equals the last observed frame.
- **Severity - threshold persistence:** apply the severity thresholds to the max depth of the
  last observed frame. No model, no training.

The training loss below is `MSE / persistence MSE`: **1.0 means exactly persistence, below 1.0
means better than persistence** (validation skill = 1 - that ratio).

In [ ]:
def persistence_mse(examples):
    return float(np.mean([np.mean((y - x[-1, ..., 0][None]) ** 2) for x, y, _ in examples]))


PERSISTENCE_MSE = max(persistence_mse(train_data), 1e-12)
print(f"persistence MSE on training windows: {PERSISTENCE_MSE:.3e} m^2  (RMSE {np.sqrt(PERSISTENCE_MSE) * 1000:.1f} mm)")

val_baseline_acc = float(np.mean([w[4] == w[3] for w in val_windows]))
print(f"severity baseline (threshold the last observed frame): {val_baseline_acc:.1%} on the balanced validation windows")
print(f"severity chance level on the balanced validation windows: 25.0%")

## Build the model and compiled train / predict steps

Combined depth + severity loss through the ONE shared encoder (Stage 3's verified architecture),
`@tf.function`-compiled for GPU speed.

In [ ]:
import model as flood_model
from model import decode_horizon
from capsule_layers import margin_loss
import tensorflow as tf

tf.keras.utils.set_random_seed(SEED)
encoder = flood_model.build_encoder(hidden_dim=128, dropout=0.3)
decoder = flood_model.build_decoder_step(hidden_dim=128, dropout=0.3)
capsule_head = flood_model.build_capsule_head(return_lengths=True)  # raw capsule lengths -> margin loss
# Constant-LR batch-size-1 Adam made epoch-to-epoch validation swing by several points in local runs
# (and left the final epoch's weights to luck). Cosine decay makes the LAST weights the stable ones.
lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
    LEARNING_RATE, decay_steps=EPOCHS * len(train_data), alpha=0.02)
optimizer = flood_model.OPTIMIZERS["adam"](learning_rate=lr_schedule, clipnorm=CLIPNORM)
all_vars = encoder.trainable_variables + decoder.trainable_variables + capsule_head.trainable_variables

X_SPEC = tf.TensorSpec(shape=[1, T_IN, PATCH_SIZE, PATCH_SIZE, 3], dtype=tf.float32)
Y_SPEC = tf.TensorSpec(shape=[HORIZON, PATCH_SIZE, PATCH_SIZE], dtype=tf.float32)
S_SPEC = tf.TensorSpec(shape=[1], dtype=tf.int32)


@tf.function(input_signature=[X_SPEC, Y_SPEC, S_SPEC])
def train_step(x, depth_targets, severity_label):
    with tf.GradientTape() as tape:
        enc_out = encoder(x, training=True)
        frames = decode_horizon(decoder, enc_out, x, HORIZON, training=True)
        depth_loss = tf.reduce_mean(tf.square(frames[0] - depth_targets)) / PERSISTENCE_MSE
        severity_lengths = capsule_head(enc_out[2][:, -1], training=True)
        severity_loss = margin_loss(severity_label, severity_lengths)
        loss = depth_loss + severity_loss
    grads = tape.gradient(loss, all_vars)
    optimizer.apply_gradients((g, v) for g, v in zip(grads, all_vars) if g is not None)
    return depth_loss, severity_loss


@tf.function(input_signature=[X_SPEC])
def predict_step(x):
    enc_out = encoder(x, training=False)
    frames = decode_horizon(decoder, enc_out, x, HORIZON, training=False)
    severity_lengths = capsule_head(enc_out[2][:, -1], training=False)
    return frames[0], tf.argmax(severity_lengths[0])


def predict(input_seq):
    frames, sev = predict_step(tf.constant(input_seq[np.newaxis, ...], dtype=tf.float32))
    return frames.numpy(), int(sev)

## Train

Each epoch prints validation **depth skill vs persistence** (positive = beats it) and severity
accuracy on patches the model has never seen. Epoch 0 is a sanity check: the untrained model
*is* persistence, so its skill must be ~0.

In [ ]:
import time


def val_scores(examples):
    mses, pers, correct = [], [], 0
    for x, y, s in examples:
        frames, pred_sev = predict(x)
        mses.append(np.mean((frames - y) ** 2))
        pers.append(np.mean((x[-1, ..., 0][None] - y) ** 2))
        correct += int(pred_sev == s)
    return 1 - np.mean(mses) / np.mean(pers), correct / len(examples)


skill0, acc0 = val_scores(val_data)
print(f"epoch  0 (untrained)  val_depth_skill={skill0:+.4f}  val_sev_acc={acc0:.1%}   <- skill must be ~0")
assert abs(skill0) < 1e-3, "untrained model should equal persistence"

def augment(x, y, rng):
    """One random element of the 8 flips/rotations of the square patch, applied identically to the input
    frames and the depth targets. A mirrored or rotated flood scene is an equally valid scene (terrain,
    water and rain are transformed together) and the severity label does not change."""
    k, flip = int(rng.integers(4)), bool(rng.integers(2))
    x, y = np.rot90(x, k, axes=(1, 2)), np.rot90(y, k, axes=(1, 2))
    if flip:
        x, y = x[:, :, ::-1], y[:, :, ::-1]
    return np.ascontiguousarray(x), np.ascontiguousarray(y)


aug_rng = np.random.default_rng(SEED + 1)
history = [(0, 1.0, float("nan"), skill0, acc0)]
for epoch in range(1, EPOCHS + 1):
    t0 = time.perf_counter()
    dls, sls = [], []
    for i in np.random.permutation(len(train_data)):
        input_seq, depth_targets, severity = train_data[i]
        if AUGMENT:
            input_seq, depth_targets = augment(input_seq, depth_targets, aug_rng)
        dl, sl = train_step(
            tf.constant(input_seq[np.newaxis, ...], dtype=tf.float32),
            tf.constant(depth_targets, dtype=tf.float32),
            tf.constant([severity], dtype=tf.int32),
        )
        dls.append(float(dl)); sls.append(float(sl))
    skill, acc = val_scores(val_data)
    history.append((epoch, np.mean(dls), np.mean(sls), skill, acc))
    print(f"epoch {epoch:2d}/{EPOCHS}  train_depth/persist={np.mean(dls):.4f}  train_sev_loss={np.mean(sls):.4f}  "
          f"val_depth_skill={skill:+.4f}  val_sev_acc={acc:.1%}  ({time.perf_counter() - t0:.1f}s)")

In [ ]:
import matplotlib.pyplot as plt

ep, tr_d, tr_s, va_skill, va_acc = zip(*history)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(ep, tr_d); axes[0].axhline(1.0, color="gray", linestyle="--", label="= persistence")
axes[0].set_title("Train depth loss / persistence loss"); axes[0].legend()
axes[1].plot(ep, va_skill); axes[1].axhline(0.0, color="gray", linestyle="--", label="= persistence")
axes[1].set_title("Val depth skill (>0 beats persistence)"); axes[1].legend()
axes[2].plot(ep, va_acc, label="capsule")
axes[2].axhline(0.25, color="gray", linestyle="--", label="chance")
axes[2].axhline(val_baseline_acc, color="green", linestyle="--", label="threshold baseline")
axes[2].set_title("Val severity accuracy"); axes[2].legend()
plt.tight_layout(); plt.show()

**Reading the severity curve.** The capsule head is trained with the CapsNet paper's *margin loss* on
capsule lengths (true class above 0.9, others below 0.1). The first version used softmax
cross-entropy on the lengths; because lengths are bounded in [0, 1), that softmax can never exceed
~0.475 for one of 4 classes, and locally the head collapsed to predicting a single class.

Honest caveats that remain regardless of loss:
- The label is "max depth in the patch" - exactly what the threshold baseline measures - so a capsule head
  that merely matches the baseline adds nothing a threshold doesn't. The interesting question is the windows
  where the class *changes*; the evaluation cell reports the capsule's accuracy there separately.
- **Overfitting.** With only ~46 training patches the head can memorise them (in the first Colab run the
  training margin loss fell to 0.026 while validation accuracy stayed flat). `AUGMENT` (random flips and
  rotations of each training window) was added to attack this. In local A/B tests on two seeds it raised
  severity accuracy both times (+18 and +13.5 points) but, on the second seed, training destabilised late and
  depth skill fell (UK +0.21 -> +0.04). It is therefore **off by default**; set `AUGMENT = True` to try it.
  Details and the caveats are in `PROJECT_LOG.md`.
- Whether capsule routing dilutes a single deep pixel is a suspicion, not something that was tested.

## Evaluation

RMSE, NSE, r, CSI (flood-extent skill; a pixel is wet above 0.1 m). **Skill** = 1 - (model RMSE /
persistence RMSE)^2; positive means the model beats persistence.

**Comparison with the FloodCastBench paper's U-Net / FNO / FNO+:** take the published numbers from the
paper's own tables. They are deliberately not typed in here - they were not independently verified,
and the paper's evaluation setup (whole scenes, its own horizons) differs from this patch-based one.

In [ ]:
def compute_metrics(pred, obs, wet_threshold=0.1):
    p, o = pred.flatten(), obs.flatten()
    rmse = float(np.sqrt(np.mean((p - o) ** 2)))
    denom = np.sum((o - o.mean()) ** 2)
    nse = float(1 - np.sum((o - p) ** 2) / denom) if denom > 0 else float("nan")
    r = float(np.corrcoef(p, o)[0, 1]) if p.std() > 0 and o.std() > 0 else float("nan")
    pw, ow = p > wet_threshold, o > wet_threshold
    hits, misses, false_alarms = int(np.sum(pw & ow)), int(np.sum(~pw & ow)), int(np.sum(pw & ~ow))
    denom_csi = hits + misses + false_alarms
    csi = hits / denom_csi if denom_csi > 0 else float("nan")
    return {"RMSE": rmse, "NSE": nse, "r": r, "CSI": csi}


def skill(model_m, base_m):
    return float(1 - (model_m["RMSE"] / base_m["RMSE"]) ** 2)


def evaluate_depth(examples):
    preds, obs, persistence = [], [], []
    for input_seq, depth_targets, _ in examples:
        frames, _ = predict(input_seq)
        preds.append(frames)
        obs.append(depth_targets)
        persistence.append(np.repeat(input_seq[-1, ..., 0][None], HORIZON, axis=0))
    obs = np.concatenate(obs)
    return compute_metrics(np.concatenate(preds), obs), compute_metrics(np.concatenate(persistence), obs)


def severity_report(examples, baseline_labels):
    y_true = np.array([s for _, _, s in examples])
    y_pred = np.array([predict(x)[1] for x, _, _ in examples])
    cm = np.zeros((4, 4), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    present = [k for k in range(4) if (y_true == k).any()]
    recalls = [cm[k, k] / cm[k].sum() for k in present]
    baseline = np.array(baseline_labels)
    base_right = baseline == y_true
    return {"capsule accuracy": float((y_true == y_pred).mean()),
            "capsule balanced accuracy": float(np.mean(recalls)),
            "threshold-baseline accuracy": float(base_right.mean()),
            "n baseline right": int(base_right.sum()),
            "capsule acc where baseline is right": float((y_pred == y_true)[base_right].mean()) if base_right.any() else float("nan"),
            "n baseline wrong": int((~base_right).sum()),
            "capsule acc where baseline is wrong": float((y_pred == y_true)[~base_right].mean()) if (~base_right).any() else float("nan"),
            "per-class recall": {SEVERITY_CLASSES[k]: float(cm[k, k] / cm[k].sum()) for k in present},
            "confusion": cm}


au_model, au_base = evaluate_depth(val_data)
print("Depth - Australia held-out patches (class-balanced windows):")
print(f"  {'metric':6s} {'model':>10s} {'persistence':>13s}")
for k in au_model:
    print(f"  {k:6s} {au_model[k]:10.5f} {au_base[k]:13.5f}")
print(f"  skill vs persistence = {skill(au_model, au_base):+.4f}   (positive = beats persistence)")

sev = severity_report(val_data, [w[4] for w in val_windows])
print("\nSeverity - Australia held-out patches (balanced, chance = 25%):")
for k in ("capsule accuracy", "capsule balanced accuracy", "threshold-baseline accuracy"):
    print(f"  {k:28s} {sev[k]:.1%}")
print(f"  where the threshold baseline is right (n={sev['n baseline right']}): capsule {sev['capsule acc where baseline is right']:.1%}")
wrong_acc = sev['capsule acc where baseline is wrong']
print(f"  where the threshold baseline is wrong (n={sev['n baseline wrong']}; the class changes over the forecast): "
      f"capsule {'n/a' if sev['n baseline wrong'] == 0 else format(wrong_acc, '.1%')}  (baseline 0% by definition)")
print("  per-class recall:", {k: f"{v:.0%}" for k, v in sev["per-class recall"].items()})
print("  confusion matrix (rows = true none/minor/moderate/severe, cols = capsule prediction):")
print(sev["confusion"])

## Grad-CAM on a real trained example

Locally (9 training examples) the heatmap was a smooth positional gradient unrelated to the flood
channel. The decoder is now persistence-anchored, so this heatmap explains what drives the model's
predicted **change** from the last frame, in whichever direction the model predicts it (Grad-CAM's
ReLU would otherwise blank the whole map whenever the model predicts net drying - seen in a local run).
The number to read is the gap between the two means: a clearly higher mean on wet pixels is what a
meaningful explanation looks like.

In [ ]:
from grad_cam import grad_cam

heat_idx = int(np.argmax([x[-1, ..., 0].mean() for x, _, _ in val_data]))  # wettest validation example
sample_input, _, _ = val_data[heat_idx]
heatmap, net_change_m = grad_cam(encoder, decoder, sample_input, horizon_step=0)
n_pixels = PATCH_SIZE * PATCH_SIZE
print(f"model predicts a net {'RISE' if net_change_m >= 0 else 'FALL'} of "
      f"{abs(net_change_m) / n_pixels * 1000:.2f} mm in mean depth at the first forecast step "
      f"-> the map shows evidence for that direction")

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
last_depth = sample_input[-1, ..., 0]
axes[0].imshow(last_depth, cmap="Blues"); axes[0].set_title("last observed depth"); axes[0].axis("off")
axes[1].imshow(heatmap, cmap="jet"); axes[1].set_title("Grad-CAM heatmap"); axes[1].axis("off")
axes[2].imshow(last_depth, cmap="Blues"); axes[2].imshow(heatmap, cmap="jet", alpha=0.5)
axes[2].set_title("overlay"); axes[2].axis("off")
plt.tight_layout(); plt.show()

wet = last_depth > last_depth.mean()
print(f"mean heatmap on above-average-depth pixels: {heatmap[wet].mean():.4f}")
print(f"mean heatmap on below-average-depth pixels: {heatmap[~wet].mean():.4f}")

# One window is an anecdote: in local runs this same number flipped from 0.34-vs-0.02 to 0.09-vs-0.18 on
# different windows. Measure it over many evenly-spaced validation windows that actually contain water.
picked = np.linspace(0, len(val_data) - 1, min(40, len(val_data))).astype(int)
wet_heat, dry_heat = [], []
for i in picked:
    x_i, _, _ = val_data[i]
    d_i = x_i[-1, ..., 0]
    if d_i.max() <= 0.1:          # no water to compare against
        continue
    hm_i, _ = grad_cam(encoder, decoder, x_i, horizon_step=0)
    w_i = d_i > d_i.mean()
    if hm_i.max() == 0 or w_i.all() or not w_i.any():
        continue
    wet_heat.append(hm_i[w_i].mean()); dry_heat.append(hm_i[~w_i].mean())
wet_heat, dry_heat = np.array(wet_heat), np.array(dry_heat)
if len(wet_heat):
    print(f"across {len(wet_heat)} validation windows with water: heat is higher on wet than on dry pixels in "
          f"{(wet_heat > dry_heat).mean():.0%} of them (50% would be chance); "
          f"median wet/dry ratio {np.median(wet_heat / np.maximum(dry_heat, 1e-9)):.1f}x")

## Symbolic reasoning on held-out Australian patches

The rule thresholds were derived from Australia at 30m using the native 5-minute step; depth-rate is
computed per hour of real time, so it stays comparable at this 30-minute stride. This demo is
Australia only (the function warns if pointed at any other resolution).

In [ ]:
from predict_severity import predict_severity_with_reasoning

for (row, col, start_idx, _sev, _base) in val_windows[:8]:
    result = predict_severity_with_reasoning(
        encoder, decoder, capsule_head, rd_au, row, col, PATCH_SIZE, start_idx, T_IN, HORIZON, TIME_STRIDE
    )
    f = result["features"]
    print(f"true={result['true_severity']:9s} raw={result['raw_severity']:9s} reasoned={result['reasoned_severity']:9s}")
    print(f"  depth_rate={f['depth_rate_m_per_hr']:.4f} m/hr  rainfall={f['rainfall_mm']:.2f} mm  "
          f"slope={f['slope_pct']:.2f}%  drainage={f['drainage_proximity_m']:.0f} m")
    print(f"  {result['justification']}\n")

## Does the symbolic layer improve accuracy?

The demo above shows 8 windows, which proves nothing about accuracy. This runs the full capsule -> rules
pipeline on **every** balanced validation window and counts what the rules did to the capsule's answer:
how many they changed, how many of those changes fixed a wrong answer, and how many broke a right one.

Caveat: the rule thresholds were derived from real Australian windows before this notebook existed, so they
are not tuned on this validation set, but they were not held out from Australian patches in general either.

In [ ]:
from predict_severity import predict_severity_with_reasoning

raw_ok = reasoned_ok = changed = fixed = broken = 0
for (row, col, start_idx, _sev, _base) in val_windows:
    res = predict_severity_with_reasoning(
        encoder, decoder, capsule_head, rd_au, row, col, PATCH_SIZE, start_idx, T_IN, HORIZON, TIME_STRIDE)
    true, raw, reasoned = res["true_severity"], res["raw_severity"], res["reasoned_severity"]
    raw_ok += raw == true
    reasoned_ok += reasoned == true
    if raw != reasoned:
        changed += 1
        fixed += reasoned == true      # wrong -> right
        broken += raw == true          # right -> wrong
n_val = len(val_windows)
print(f"capsule alone      : {raw_ok / n_val:.1%}")
print(f"capsule + rules    : {reasoned_ok / n_val:.1%}")
print(f"rules changed the class in {changed} of {n_val} windows: {fixed} fixed a wrong answer, {broken} broke a right one, "
      f"{changed - fixed - broken} moved one wrong answer to another wrong class")

## Visualization

Attaches the region's real CRS and affine transform (from the archive's world files - the raw depth
TIFFs carry none) to a predicted depth grid and renders it on a real basemap.

**Basemap choice (learned the hard way):** folium's default OpenStreetMap tiles do not work in Colab -
OSM's servers reject requests that carry no `Referer` header, which Colab's output frame does not send,
and answer every tile with a "403 Access blocked" picture. CartoDB's free tiles now stamp "API KEY
REQUIRED" across the map. Esri World Imagery returned real tiles with and without a referrer, so it is
used here. A static plot with no network dependency follows the interactive map either way.

Scope: the world files describe the **30m base grid**, so this is valid for the 30m depth rasters (Australia, UK).
The 60m / 480m depth rasters sit on coarser cells (scale 2 / 16 with a crop offset, `LAYER_GEOMETRY`), so the transform would
need that scale and offset applied.

The world-file origin is the *centre* of the upper-left pixel; an affine transform wants that pixel's
*corner*, so the origin is shifted by half a pixel.

In [ ]:
import re
from rasterio.transform import Affine
from rasterio.warp import transform_bounds
import folium


def utm_zone_to_epsg(zone_str):
    m = re.search(r"Zone_(\d+)([NS])", zone_str)
    return (32600 if m.group(2) == "N" else 32700) + int(m.group(1))


def load_spatial_reference(data_root, region_name):
    with open(f"{data_root}/Relevant data/Georeferenced files/Spatial_References.txt") as f:
        for line in f:
            if region_name.lower() in line.lower():
                return utm_zone_to_epsg(line.split(":")[1].strip())
    raise ValueError(f"No spatial reference entry for {region_name}")


def load_world_file(data_root, region_name):
    with open(f"{data_root}/Relevant data/Georeferenced files/{region_name}.tfw") as f:
        a, _d, _b, e, c, fy = (float(x) for x in f.read().split())
    return a, e, c, fy  # pixel width, pixel height (negative), x/y of upper-left pixel CENTRE


row, col, start_idx, _, _ = val_windows[heat_idx]  # same (wettest) example the Grad-CAM cell used
input_seq, _, _, _ = rd_au.load_patch_sequence(row, col, PATCH_SIZE, start_idx, T_IN, HORIZON, TIME_STRIDE)
pred = predict(input_seq)[0][0]  # first forecast step

epsg = load_spatial_reference(DATA_ROOT, "Australia")
pixel_w, pixel_h, cx, cy = load_world_file(DATA_ROOT, "Australia")
corner_x, corner_y = cx - pixel_w / 2, cy - pixel_h / 2
transform = Affine(pixel_w, 0, corner_x + col * pixel_w, 0, pixel_h, corner_y + row * pixel_h)

out_path = "predicted_depth_georeferenced.tif"
with rasterio.open(out_path, "w", driver="GTiff", height=pred.shape[0], width=pred.shape[1], count=1,
                   dtype="float32", crs=f"EPSG:{epsg}", transform=transform) as dst:
    dst.write(pred.astype("float32"), 1)
print(f"wrote {out_path} with CRS EPSG:{epsg}")

with rasterio.open(out_path) as src:
    lon_min, lat_min, lon_max, lat_max = transform_bounds(src.crs, "EPSG:4326", *src.bounds)
print(f"patch spans lat {lat_min:.4f}..{lat_max:.4f}, lon {lon_min:.4f}..{lon_max:.4f}")

# scrollWheelZoom=False: otherwise scrolling down the notebook with the pointer over the map zooms it away.
m = folium.Map(location=[(lat_min + lat_max) / 2, (lon_min + lon_max) / 2], zoom_start=13, tiles=None,
               scrollWheelZoom=False)
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Tiles &copy; Esri &mdash; Source: Esri, Maxar, Earthstar Geographics, and the GIS User Community",
    name="Esri World Imagery", max_zoom=19,
).add_to(m)
folium.raster_layers.ImageOverlay(
    image=np.clip(np.sqrt(pred / (pred.max() + 1e-9)), 0, 1),  # sqrt so shallow water is visible next to deep
    bounds=[[lat_min, lon_min], [lat_max, lon_max]],
    opacity=0.6, colormap=lambda x: (0, 0, 1, x),
).add_to(m)
folium.Rectangle(bounds=[[lat_min, lon_min], [lat_max, lon_max]], color="red", weight=2, fill=False,
                 tooltip="predicted patch").add_to(m)
m.fit_bounds([[lat_min, lon_min], [lat_max, lon_max]])

# static version - needs no tile server. The axes are lon/lat of the (UTM) patch's bounding box.
fig, ax = plt.subplots(figsize=(5, 5))
im = ax.imshow(pred, cmap="Blues", extent=[lon_min, lon_max, lat_min, lat_max])
ax.set_xlabel("longitude"); ax.set_ylabel("latitude"); ax.set_title("predicted depth, first forecast step (m)")
fig.colorbar(im, ax=ax, shrink=0.8); plt.tight_layout(); plt.show()
m

## Cross-regional generalization (Australia -> UK)

The model never saw the UK. Depth is scored against the same persistence baseline. The Australia rows
use a **uniformly sampled** (not class-balanced) set so the two regions are like-for-like.

**Severity is not scored on the UK, on purpose:** every UK window is "severe" under this label
definition, so an "always severe" model would look perfect. The census below shows that directly.

In [ ]:
rd_uk = regions["UK"]
uk_positions = valid_positions(rd_uk, PATCH_SIZE, DEM_NODATA_STD)
uk_starts = list(range(0, rd_uk.max_start_index(T_IN, HORIZON, TIME_STRIDE) + 1, UK_START_STEP))
uk_windows = scan_windows(rd_uk, uk_positions, uk_starts, PATCH_SIZE, T_IN, HORIZON, TIME_STRIDE)
uk_counts = {c: sum(1 for w in uk_windows if w[3] == i) for i, c in enumerate(SEVERITY_CLASSES)}
print(f"UK: {len(uk_positions)} patches x {len(uk_starts)} times = {len(uk_windows)} windows")
print("UK severity census:", uk_counts, "-> severity is NOT assessable on the UK")

uk_data = load_examples(rd_uk, uk_windows)
uk_model, uk_base = evaluate_depth(uk_data)
au_u_model, au_u_base = evaluate_depth(au_uniform_data)

print("\nDepth, uniformly-sampled windows (like-for-like):")
print(f"  {'':24s} {'RMSE':>9s} {'NSE':>9s} {'r':>9s} {'CSI':>9s}")
for label, mtr in [("Australia model", au_u_model), ("Australia persistence", au_u_base),
                   ("UK model (held out)", uk_model), ("UK persistence", uk_base)]:
    print(f"  {label:24s} {mtr['RMSE']:9.5f} {mtr['NSE']:9.5f} {mtr['r']:9.5f} {mtr['CSI']:9.5f}")

sk_au, sk_uk = skill(au_u_model, au_u_base), skill(uk_model, uk_base)
print("\n=== SUMMARY ===")
print(f"Depth, Australia held-out patches : skill vs persistence = {sk_au:+.4f}  "
      f"({'beats' if sk_au > 0 else 'does NOT beat'} persistence)")
print(f"Depth, UK (never seen)            : skill vs persistence = {sk_uk:+.4f}  "
      f"({'beats' if sk_uk > 0 else 'does NOT beat'} persistence)")
print(f"Severity, Australia held-out      : capsule {sev['capsule accuracy']:.1%} vs threshold baseline "
      f"{sev['threshold-baseline accuracy']:.1%} (chance 25%)")
print("A cross-regional transfer claim is supported only if the UK skill is positive.")

## What does the trained model actually use?

A positive skill does not say *why*. This removes one input at a time from the trained model's test data and re-measures
skill (and severity accuracy). A large drop means the model relies on that input; a drop near zero means it does not - and
then a claim that the model exploits it (for example "the LSTM captures temporal context") is unsupported.

In the audited local runs the answer was stark: removing **rainfall** took skill from about +0.22 to about 0; removing
terrain, scrambling the order of the depth history, or freezing it changed skill by less than 0.01. The forecast is
essentially *last frame + current rainfall*. (Test-time removal measures reliance, not what a differently-trained model could
learn; and the severity head *does* use terrain.) Also reported: skill at each lead time.

In [ ]:
def all_preds(examples, transform=None):
    preds, obs, pers, ok = [], [], [], 0
    for x, y, s in examples:
        frames, sev = predict(x if transform is None else transform(x))
        preds.append(frames); obs.append(y)
        pers.append(np.repeat(x[-1, ..., 0][None], HORIZON, axis=0))  # persistence always uses the TRUE last frame
        ok += int(sev == s)
    return np.stack(preds), np.stack(obs), np.stack(pers), ok / len(examples)


def skill_np(p, o, q):
    return 1.0 - np.mean((p - o) ** 2) / np.mean((q - o) ** 2)


def no_rain(x):
    y = x.copy(); y[..., 2] = 0.0; return y


def flat_terrain(x):
    y = x.copy(); y[..., 1] = 0.0; return y  # the DEM is made patch-relative inside the model, so zero = flat


def scrambled_history(x):
    return np.concatenate([x[:-1][::-1], x[-1:]], axis=0)  # reverse the first frames, keep the last (the anchor)


def frozen_history(x):
    return np.repeat(x[-1:], x.shape[0], axis=0)  # every history frame = the last one: no temporal information


rows_skill = []
print(f"{'input removed at test time':32s} {'AU skill':>9s} {'UK skill':>9s} {'severity acc':>13s}")
for name, fn in [("none (normal)", None), ("rainfall", no_rain), ("terrain (flat DEM)", flat_terrain),
                 ("history order scrambled", scrambled_history), ("history frozen (no dynamics)", frozen_history)]:
    pa, oa, qa, acc_a = all_preds(val_data, fn)
    pu, ou, qu, _ = all_preds(uk_data, fn)
    rows_skill.append(skill_np(pa, oa, qa))
    print(f"{name:32s} {skill_np(pa, oa, qa):+9.3f} {skill_np(pu, ou, qu):+9.3f} {acc_a:12.1%}")
if max(rows_skill) - min(rows_skill) < 5e-4:
    print("WARNING: skill is IDENTICAL whichever input is removed -> the network is input-blind (it collapsed to a constant "
          "offset during training). These numbers are NOT a result; rerun with a different SEED or a lower LEARNING_RATE.")

print("skill by lead time (30-minute steps):")
for name, ex in [("Australia held-out", val_data), ("UK (never seen)", uk_data)]:
    p, o, q, _ = all_preds(ex)
    print(f"  {name:20s}", "  ".join(f"{1 - np.mean((p[:, k] - o[:, k]) ** 2) / np.mean((q[:, k] - o[:, k]) ** 2):+.3f}"
                                    for k in range(HORIZON)))

## Save the trained model

In [ ]:
encoder.save_weights("encoder.weights.h5")
decoder.save_weights("decoder.weights.h5")
capsule_head.save_weights("capsule_head.weights.h5")
print("saved. To persist beyond this session: from google.colab import drive; drive.mount('/content/drive'), "
      "then copy these files (and predicted_depth_georeferenced.tif) to your Drive.")